# CozyDay P1 A8 — Stage 1
**Owner: Lin Gao** | Step 1.1 Knowledge Base and Step 1.2 Chunking only. Stages 2–4 will be added by teammates after Stage 1 is merged.

## 1. Load and validate the knowledge base
30 original English paragraphs with stable IDs, titles, topics, and bodies. All filesystem paths are relative to this notebook / module directory.

In [1]:
from chunking import (load_knowledge_base, validate_knowledge_base,
                      fixed_length_chunks, overlapping_paragraph_chunks, hybrid_topic_chunks,
                      build_all_chunks, validate_chunks)
records = load_knowledge_base()
validation = validate_knowledge_base(records)
print(validation)
print("First paragraph:", records[0]["id"], records[0]["title"])

{'paragraphs': 30, 'total_words': 3111, 'min_words': 97, 'max_words': 112, 'unique_ids': 30}
First paragraph: P001 Creating an actionable task


## 2. Three deterministic chunking strategies
**Fixed:** whole-sentence packing to 130 words, flushing at fine-topic boundaries. **Overlap:** adjacent whole-paragraph pairs inside documented related sections, stride 1. **Hybrid:** nonoverlapping whole paragraphs with identical fine topics, soft cap 230 words.

All examples below show complete chunk text. See `rag_analysis.md` for the semantic section mapping and soft-cap exceptions.

In [2]:
chunks_by_strategy = build_all_chunks(records)
for strategy, chunks in chunks_by_strategy.items():
    print(f"{strategy}: {len(chunks)} chunks")
    example = chunks[0]
    print("ID:", example["chunk_id"], "| Topic:", example["topic"])
    print("Source IDs:", example["source_ids"], "| Words:", example["word_count"])
    print("Complete example:\n" + example["text"] + "\n")
print("Actual overlapping windows in Task Organization:")
for chunk in chunks_by_strategy["overlap"][:3]:
    print(chunk["chunk_id"], chunk["source_ids"], chunk["topic"])

fixed: 30 chunks
ID: FIXED_001 | Topic: Task Creation
Source IDs: ['P001'] | Words: 112
Complete example:
A useful task describes one clear outcome rather than a vague intention. For example, replace “work on biology” with “summarize chapter four and answer the practice questions.” When preparing a task in CozyDay, give it a short descriptive title, optional notes, an estimated duration, and a realistic due date if one exists. Record the next physical action in the notes when the work feels overwhelming. Large assignments should become several smaller tasks, each with an observable completion criterion. Review a newly entered task to make sure its wording still makes sense when seen without the surrounding conversation. An organized task list supports planning, but creating tasks does not replace doing the work.

overlap: 19 chunks
ID: OVERLAP_001 | Topic: Task Creation
Source IDs: ['P001', 'P002'] | Words: 216
Complete example:
A useful task describes one clear outcome rather than a v

## 3. Automated checks and Stage 2 interface
Validation checks text preservation, exact source provenance, metadata, semantic boundaries, and coverage. The standalone regression suite is run with `python test_stage1.py`. Stage 2 embeds `chunk["text"]` and preserves the other fields. Full-corpus size validation is separate from chunking so a small scaling subset remains supported.

In [3]:
for strategy, chunks in chunks_by_strategy.items():
    validate_chunks(records, chunks, strategy)
print("Stage 1 checks PASSED")
print("Chunk object keys:", list(chunks_by_strategy["fixed"][0]))
print("Five-paragraph scaling subset:",
      {name: len(chunks) for name, chunks in build_all_chunks(records[:5]).items()})

Stage 1 checks PASSED
Chunk object keys: ['chunk_id', 'strategy', 'source_ids', 'topic', 'text', 'word_count']
Five-paragraph scaling subset: {'fixed': 5, 'overlap': 4, 'hybrid': 3}


# CozyDay P1 A8 -- Stage 2
**Owner: Rishabh Puri** | Step 1.3 Embedding Pipeline, Step 1.4 Retrieval System, Step 1.5 Generation Pipeline, Step 2.1 Test Queries.

Builds directly on Stage 1's frozen interface above (`records`, `chunks_by_strategy`). The pipeline logic lives in five small modules next to this notebook -- `embedding.py`, `retrieval.py`, `generation.py`, `queries.py`, `rag.py` -- so Stage 3 and Stage 4 can import the same functions instead of re-deriving them.

## 4. Embedding pipeline (Step 1.3)
Three sentence-transformers models fixed by the team's shared contract -- `all-MiniLM-L6-v2` (384d), `all-mpnet-base-v2` (768d), `bge-large-en-v1.5` (1024d) -- loaded once, with device selection (cuda -> mps -> cpu) and each model's *actual* output dimension checked against the expected dimensionality. Every chunk from all three Stage 1 strategies is embedded for all three models (9 combinations total) and cached to `results/embeddings/` so re-running this cell does not recompute vectors that are already on disk.

In [4]:
from embedding import load_all_embedding_models, describe_models, embed_all_combinations

model_infos = load_all_embedding_models()
print("Embedding models loaded:")
for row in describe_models(model_infos):
    print(" ", row)

embeddings = embed_all_combinations(model_infos, chunks_by_strategy)
print("\nEmbedded all 3 models x 3 chunking strategies = 9 combinations:")
for model_key, by_strategy in embeddings.items():
    for strategy, result in by_strategy.items():
        status = "cached" if result["from_cache"] else f"{result['embed_time_seconds']:.3f}s"
        print(f"  {model_key:7s} {strategy:8s} {result['vectors'].shape}  {status}")


/Users/rishabhpuri/PycharmProjects/3_CozyDay/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1028 17:41:26.296000 36613 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|                                                                                                                                 | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10036.08it/s]

Loading weights:   0%|                                                                                                                                 | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 8781.53it/s]

Loading weights:   0%|                                                                                                                                 | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 10047.25it/s]

Embedding models loaded:
  {'key': 'small', 'model_name': 'sentence-transformers/all-MiniLM-L6-v2', 'dimension': 384, 'device': 'mps', 'load_time_seconds': 0.6769464169628918}
  {'key': 'medium', 'model_name': 'sentence-transformers/all-mpnet-base-v2', 'dimension': 768, 'device': 'mps', 'load_time_seconds': 0.6906309998594224}
  {'key': 'large', 'model_name': 'BAAI/bge-large-en-v1.5', 'dimension': 1024, 'device': 'mps', 'load_time_seconds': 1.5578322501387447}

Embedded all 3 models x 3 chunking strategies = 9 combinations:
  small   fixed    (30, 384)  cached
  small   overlap  (19, 384)  cached
  small   hybrid   (15, 384)  cached
  medium  fixed    (30, 768)  cached
  medium  overlap  (19, 768)  cached
  medium  hybrid   (15, 768)  cached
  large   fixed    (30, 1024)  cached
  large   overlap  (19, 1024)  cached
  large   hybrid   (15, 1024)  cached


## 5. Retrieval system (Step 1.4)
`retrieve(query, model_info, strategy, embeddings, chunk_lookup, top_k)` is the one consistent entry point used for every experiment in this assignment. It embeds the query with the requested model, ranks cached chunk vectors with scikit-learn's `cosine_similarity` (the searchable structure the assignment asks for), and returns chunk IDs, complete chunk text, similarity scores, and both the embedding- and search-latency separately.

In [5]:
from retrieval import build_chunk_lookup, retrieve
from queries import TEST_QUERIES, validate_test_queries

validate_test_queries(records)
chunk_lookup = build_chunk_lookup(chunks_by_strategy)
print("All 5 shared test queries' expected evidence IDs exist in the knowledge base.\n")

demo_query = TEST_QUERIES[0]
demo = retrieve(demo_query["query"], model_infos["small"], "fixed", embeddings,
                chunk_lookup, top_k=3)
print("Query:", demo_query["query"])
print("Expected evidence:", demo_query["expected_evidence_ids"])
print(f"embedding_latency={demo['embedding_latency_seconds']:.4f}s "
      f"retrieval_latency={demo['retrieval_latency_seconds']:.4f}s")
for chunk in demo["chunks"]:
    print(" ", chunk["chunk_id"], round(chunk["score"], 4), chunk["text"][:90])


All 5 shared test queries' expected evidence IDs exist in the knowledge base.

Query: I have a big semester paper due in a few weeks. How should I break it down into tasks in CozyDay?
Expected evidence: ['P002', 'P001']
embedding_latency=0.0058s retrieval_latency=0.0110s
  FIXED_002 0.6772 A semester paper can be represented as a sequence of manageable tasks: choose a topic, fin
  FIXED_003 0.6046 Categories help students see different commitments without treating every item as equally 
  FIXED_001 0.5987 A useful task describes one clear outcome rather than a vague intention. For example, repl


## 6. Five shared test queries (Step 2.1)
These five questions cover five different knowledge-base topics and every one has answerable evidence in the dataset (checked above by `validate_test_queries`). The same five queries, in this same wording, are used for every experiment in Stages 2 and 3 -- see `queries.py` for the full `expected_evidence_notes` on why each evidence paragraph answers its query.

In [6]:
for item in TEST_QUERIES:
    print(f"{item['query_id']}: {item['query']}")
    print(f"  topics: {item['topics']}")
    print(f"  expected evidence: {item['expected_evidence_ids']}\n")


Q1: I have a big semester paper due in a few weeks. How should I break it down into tasks in CozyDay?
  topics: ['Task Creation']
  expected evidence: ['P002', 'P001']

Q2: I have three assignments due this week and they all feel urgent. How do I decide what to work on first?
  topics: ['Priorities', 'Common Questions']
  expected evidence: ['P006', 'P005', 'P025']

Q3: How do I build a weekly study plan and prepare for my exam using spaced review?
  topics: ['Study Planning']
  expected evidence: ['P011', 'P012']

Q4: I want to share my group project plan with teammates. How do I do that without exposing private information?
  topics: ['Privacy']
  expected evidence: ['P016', 'P015']

Q5: How should I use my task completion analytics to write a weekly progress report?
  topics: ['Analytics', 'Reports']
  expected evidence: ['P019', 'P020', 'P021']



## 7. Generation pipeline (Step 1.5)
One locked generation model and decoding configuration for every experiment in this assignment: `Qwen/Qwen2.5-3B-Instruct`, previously evaluated as Model 9 in the team's A7 multi-model assignment, with greedy (non-sampling) decoding and a fixed `max_new_tokens`. The prompt is built from a system instruction, the retrieved context, and the user query; the system instruction explicitly tells the model to rely on the context and to acknowledge insufficient evidence rather than guess.

In [7]:
from generation import load_generation_model, describe_generation_model

generation_info = load_generation_model()
print("Generation model loaded:", describe_generation_model(generation_info))


Loading weights:   0%|                                                                                                                                 | 0/434 [00:00<?, ?it/s]

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 434/434 [00:00<00:00, 8391.31it/s]

Generation model loaded: {'model_name': 'Qwen/Qwen2.5-3B-Instruct', 'device': 'mps', 'load_time_seconds': 7.616279500070959, 'decoding_settings': {'max_new_tokens': 300, 'do_sample': False, 'num_beams': 1}}


## 8. run_rag interface and nine-combination smoke test
`run_rag(query, model_info, strategy, embeddings, chunk_lookup, generation_info, top_k)` ties embedding, retrieval, and generation together behind one interface, returning the retrieved chunks, the final answer, and embedding/retrieval/generation/total latency. The smoke test below runs the same locked generation model and settings, and the first shared query, across all nine (embedding model x chunking strategy) combinations, displaying the retrieved chunks for every combination rather than only the final answer. The full 3x3x5 = 45-row evaluation grid across all five queries is Stage 3's responsibility (Shivani); this smoke test only confirms every combination of this stage's own pipeline actually runs.

In [8]:
from rag import run_smoke_test

smoke_query = TEST_QUERIES[0]
smoke_results = run_smoke_test(smoke_query["query"], model_infos, chunks_by_strategy,
                                embeddings, chunk_lookup, generation_info)
assert len(smoke_results) == 9

for r in smoke_results:
    print(f"=== {r['model_key']} / {r['strategy']} ===")
    print("Retrieved chunks:")
    for c in r["retrieved_chunks"]:
        print("   ", c["chunk_id"], round(c["score"], 4), c["text"][:90])
    print("Final answer:", r["final_answer"])
    print(f"embed={r['embedding_latency_seconds']:.3f}s "
          f"retrieve={r['retrieval_latency_seconds']:.3f}s "
          f"generate={r['generation_latency_seconds']:.3f}s "
          f"total={r['total_latency_seconds']:.3f}s\n")

print("Stage 2 smoke test: all 9 embedding/chunking combinations completed, "
      "each with its own displayed retrieved chunks and final answer.")


=== small / fixed ===
Retrieved chunks:
    FIXED_002 0.6772 A semester paper can be represented as a sequence of manageable tasks: choose a topic, fin
    FIXED_003 0.6046 Categories help students see different commitments without treating every item as equally 
    FIXED_001 0.5987 A useful task describes one clear outcome rather than a vague intention. For example, repl
Final answer: To break down your big semester paper into manageable tasks in CozyDay, you should start by identifying key milestones and actions that contribute to completing the paper. Here's how you can structure these tasks:

1. **Choose a Topic**: Decide on the subject of your paper. This is a milestone indicating you've chosen a direction.
2. **Find Readings**: Collect relevant books, articles, and other resources. This step helps build the foundation of your research.
3. **Draft an Outline**: Create an outline to organize your thoughts and structure your paper. This outlines the main points you will cover.
4. *

## 9. Stage 2 handoff to Stage 3
Frozen Stage 2 interface for Shivani's experiments:

```python
from embedding import load_all_embedding_models, embed_all_combinations, embed_query
from retrieval import build_chunk_lookup, retrieve
from generation import load_generation_model, generate_answer, GENERATION_MODEL, DECODING_SETTINGS
from queries import TEST_QUERIES
from rag import run_rag, run_smoke_test

model_infos = load_all_embedding_models()
embeddings = embed_all_combinations(model_infos, chunks_by_strategy)  # loads from cache if present
chunk_lookup = build_chunk_lookup(chunks_by_strategy)
generation_info = load_generation_model()

# One row of the required 3 x 3 x 5 = 45-row grid:
result = run_rag(TEST_QUERIES[0]["query"], model_infos["small"], "fixed",
                  embeddings, chunk_lookup, generation_info, top_k=3)
```

`run_rag` never varies `GENERATION_MODEL` or `DECODING_SETTINGS` between calls -- Stage 3 should not either. Embeddings cache under `results/embeddings/` keyed by model name and chunk IDs, so changing `chunking.py`'s chunk contents (not expected) would invalidate the cache automatically; changing nothing means Stage 3 reuses the same cached vectors Stage 2 already verified.